# Лабораторная работа №5
## Выполнил студент группы Кесслер Алексей БВТ2306

### Оглавление
1. [Задание 1](#Задание-№1)
2. [Задание 2](#Задание-№2)
4. [Вывод](#Вывод)

> Дополнительные модули, использованные при выполнение лабораторной

In [1]:
from datetime import datetime as t
from collections import defaultdict
from collections import deque

### Задание №1
Реализовать методы поиска подстроки в строке. Добавить возможность ввода строки и подстроки с клавиатуры. Предусмотреть возможность существования пробела. Реализовать возможность выбора опции чувствительности или нечувствительности к регистру. Оценить время работы каждого алгоритма поиска и сравнить его со временем работы стандартной функции поиска, используемой в выбранном языке программирования.

#### Алгоритм Кнута-Морриса-Пратта

In [3]:
def compute_prefix_function(pattern):
    m = len(pattern)
    pi = [0] * m 
    j = 0
    for i in range(1, m):
        while j > 0 and pattern[i] != pattern[j]:
            j = pi[j - 1]
        if pattern[i] == pattern[j]:
            j += 1
        pi[i] = j
    return pi

def kmp_search(string, pattern, case_sensitive=True):
    if not case_sensitive:
        string = string.lower()
        pattern = pattern.lower()
    n, m = len(string), len(pattern)
    pi = compute_prefix_function(pattern)
    matches = []
    j = 0 
    for i in range(n):
        while j > 0 and string[i] != pattern[j]:
            j = pi[j - 1]
        if string[i] == pattern[j]:
            j += 1
        if j == m:
            matches.append(i - m + 1)
            j = pi[j - 1]
    return matches

string = input("Введите строку: ")
pattern = input("Введите подстроку: ")
case_sensitive_input = input("Учитывать регистр? (y/n): ").strip().lower()
case_sensitive = case_sensitive_input == 'y'
start_time = t.now()
result_kmp = kmp_search(string, pattern, case_sensitive)
end_time = t.now()
kmp_time = (end_time - start_time).total_seconds()
start_time = t.now()
if case_sensitive:
    result_builtin = [i for i in range(len(string) - len(pattern) + 1) if string[i:i+len(pattern)] == pattern]
else:
    result_builtin = [i for i in range(len(string) - len(pattern) + 1) if string[i:i+len(pattern)].lower() == pattern.lower()]
end_time = t.now()
builtin_time = (end_time - start_time).total_seconds()
print("\nРезультаты:")
print(f"KMP: Индексы вхождений - {result_kmp}, Время выполнения - {kmp_time:.6f} сек")
print(f"Стандартная функция: Индексы вхождений - {result_builtin}, Время выполнения - {builtin_time:.6f} сек")


Результаты:
KMP: Индексы вхождений - [1, 5], Время выполнения - 0.000307 сек
Стандартная функция: Индексы вхождений - [1, 5], Время выполнения - 0.000183 сек


#### Упрощенный алгоритм Бойера-Мура

In [ ]:
def boyer_moore_search(main_string, substring, case_sensitive=True):
    if not case_sensitive:
        main_string = main_string.lower()
        substring = substring.lower()
    m = len(main_string)
    n = len(substring)
    if n == 0:
        return [0]
    shift_table = defaultdict(lambda: n)
    for i in range(n - 1):
        shift_table[substring[i]] = n - i - 1
    i = 0
    result = []
    while i <= m - n:
        j = n - 1
        while j >= 0 and main_string[i + j] == substring[j]:
            j -= 1
        if j < 0:
            result.append(i)
            i += shift_table[main_string[i + n - 1]]
        else:
            i += shift_table[main_string[i + n - 1]]
    return result if result else [-1]

def python_find(main_string, substring, case_sensitive=True):
    if not case_sensitive:
        main_string = main_string.lower()
        substring = substring.lower()
    return [i for i in range(len(main_string) - len(substring) + 1) if main_string[i:i + len(substring)] == substring]

main_string = input("Введите строку: ")
substring = input("Введите подстроку: ")
case_sensitive_input = input("Учитывать регистр? (y/n): ").strip().lower()
case_sensitive = case_sensitive_input == 'y'
start_time = t.now()
result_bm = boyer_moore_search(main_string, substring, case_sensitive)
end_time = t.now()
bm_time = (end_time - start_time).total_seconds()
start_time = t.now()
result_builtin = python_find(main_string, substring, case_sensitive)
end_time = t.now()
builtin_time = (end_time - start_time).total_seconds()
print("\nРезультаты:")
print(f"Бойер-Мур: Индексы вхождений - {result_bm}, Время выполнения - {bm_time:.6f} сек")
print(f"Стандартная функция: Индексы вхождений - {result_builtin}, Время выполнения - {builtin_time:.6f} сек")

### Задание №2
Написать программу, определяющую, является ли данное
расположение «решаемым», то есть можно ли из него за конечное число
шагов перейти к правильному. Если это возможно, то необходимо найти хотя
бы одно решение - последовательность движений, после которой числа будут
расположены в правильном порядке.
#### Входные данные: массив чисел, представляющий собой расстановку в
Порядке «слева направо, сверху вниз». Число 0 обозначает пустое поле.
Например, массив [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0] представляет
собой «решенную» позицию элементов.
#### Выходные данные: если решения нет, то функция должна вернуть
Пустой массив []. Если решение есть, то необходимо представить решение —
для каждого шага записывается номер передвигаемого на данном шаге
элемента. 

In [ ]:
def is_solvable(puzzle):
    flattened_puzzle = [num for num in puzzle if num != 0]
    inversions = 0
    for i in range(len(flattened_puzzle)):
        for j in range(i + 1, len(flattened_puzzle)):
            if flattened_puzzle[i] > flattened_puzzle[j]:
                inversions += 1
    grid_size = int(len(puzzle) ** 0.5)
    zero_row = puzzle.index(0) // grid_size
    if grid_size % 2 == 1:
        return inversions % 2 == 0
    else:
        return (inversions + zero_row) % 2 == 1

def solve_puzzle(start):
    if not is_solvable(start):
        return []
    n = int(len(start) ** 0.5)
    goal = list(range(1, len(start))) + [0]
    queue = deque([(start, start.index(0), [])])
    visited = set()
    visited.add(tuple(start))
    def get_neighbors(puzzle, zero_idx):
        neighbors = []
        x, y = divmod(zero_idx, n)
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < n and 0 <= ny < n:
                neighbor = puzzle[:]
                neighbor_idx = nx * n + ny
                neighbor[zero_idx], neighbor[neighbor_idx] = neighbor[neighbor_idx], neighbor[zero_idx]
                neighbors.append((neighbor, neighbor_idx))
        return neighbors
    while queue:
        current, zero_idx, path = queue.popleft()
        if current == goal:
            return path
        for neighbor, neighbor_idx in get_neighbors(current, zero_idx):
            if tuple(neighbor) not in visited:
                visited.add(tuple(neighbor))
                queue.append((neighbor, neighbor_idx, path + [current[neighbor_idx]]))
    return []

start_position = [1, 2, 3, 4, 5, 6, 7, 8, 0, 10, 11, 12, 9, 13, 14, 15]
solution = solve_puzzle(start_position)
if solution:
    print("Решение:", solution)
else:
    print("Решения нет")

### Вывод